# Sales & Demand Forecasting for Businesses Model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Load the dataset
loading the datasets and check the columns, informations

In [ ]:
df=pd.read_csv("retail_store_inventory.csv")
df.describe()
df.info()
df.head()


# Handling dataset
I check for missing values and duplicate values and after that I check whether the value are correct or not and for the "Demand Forecast" feature I get negative values which are incorrect so I try to use max of 0 and itself.

In [ ]:
#handling missing ,duplicate value and check correct values
print("Check missing values")
print(df.isnull().sum())

print("Check duplicate values")
print(df.duplicated().sum())
df["Demand Forecast"]=df["Demand Forecast"].apply(lambda x:max(x,0))
df.info()
df.isnull().sum()

# Feature Engineering
I created time based features from date and lagging and rolling features to capture past behaviour and predict feature

In [ ]:
#feature engneering for time based
df["Date"]=pd.to_datetime(df["Date"])

df["Year"]=df["Date"].dt.year
df["Quarter"]=df["Date"].dt.quarter
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week
df["Day"]=df['Date'].dt.day

df['Weekday'] = df['Date'].dt.day_name()
df["Is_Weekend"]=df["Date"].dt.weekday.apply(lambda x:1 if x>=5 else 0)

def Season(month):
    if month in [12,1,2]:
        return 'Winter'
    elif month in [3,4,5]:
        return 'Spring'
    elif month in [6,7,8]:
        return 'Summer'
    else:
        return 'Autumn'
df["Season"]=df['Month'].apply(Season)



In [ ]:
#lagging feature to capute past behaviour and to predict future
df_original=df.copy()
df = df_original.copy()  # use the copy of  your original dataset
#we sort the data for chronological order
df=df.sort_values(["Store ID","Product ID","Date"])
df=df.reset_index(drop=True)


df["Sales_lag_1"]=df.groupby(["Store ID","Product ID"])['Units Sold'].shift(1)
df["Sales_lag_7"]=df.groupby(["Store ID","Product ID"])["Units Sold"].shift(7)
df["Sales_lag_30"]=df.groupby(["Store ID","Product ID"])["Units Sold"].shift(30)

df['Rolling_mean_3'] = df.groupby(['Store ID', 'Product ID'])['Units Sold'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['Rolling_std_3'] = df.groupby(['Store ID', 'Product ID'])['Units Sold'].transform(lambda x: x.rolling(3, min_periods=1).std())

# Binary features for events
df['Is_Holiday'] = df['Holiday/Promotion'].apply(lambda x: 1 if x>0 else 0)

# Lag differences to capture sudden changes and changes for larger period
df['Sales_lag_1_diff'] = df["Units Sold"] - df['Sales_lag_1']
df['Sales_lag_1_vs_lag_7_diff'] = df['Sales_lag_1'] - df['Sales_lag_7']
df['Sales_lag_7_vs_lag_30_diff'] = df['Sales_lag_7'] - df['Sales_lag_30']

df['Promo_Impact'] = df['Discount'] * df['Holiday/Promotion']
df['Price_diff'] = df['Price'] - df['Competitor Pricing']
df['High_Discount'] = (df['Discount'] > 0.2).astype(int)

# Backfill missing lag features
lag_features = ['Sales_lag_1', 'Sales_lag_7', 'Sales_lag_30', 'Rolling_mean_3', 'Rolling_std_3']
df[lag_features] = df[lag_features].fillna(method='bfill')  # or method='ffill'


# Input features and Target
I created input features from the features and I set target="Units Sold"

In [ ]:
#input features and target
target="Units Sold"
input_features=["Sales_lag_1", "Sales_lag_7", "Sales_lag_30", "Rolling_mean_3", "Rolling_std_3",
                'Sales_lag_1_diff', "Sales_lag_1_vs_lag_7_diff", 'Sales_lag_7_vs_lag_30_diff',
                "Price", "Competitor Pricing", 'Price_diff', "Discount", 'High_Discount',
                "Holiday/Promotion", 'Promo_Impact',
                "Month", "Is_Weekend", "Is_Holiday",
                ]

X=df[input_features]
y=df[target]


# Train and Test dataset split
I use date as a splitting mechanism since the dataset is time- series.

In [ ]:
#split the dataset to test and training dataset based on Date
split_date='2023-10-01'
train=df[df["Date"]<=split_date]
test=df[df["Date"]>split_date]
X_train=train[input_features]
y_train=train[target]

X_test=test[input_features]
y_test=test[target]
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)


# Training the dataset and Predict the test dataset
I use the random forest regression algorithm to train the the dataset using scikit learn.


In [ ]:
#training and predicting
from sklearn.ensemble import RandomForestRegressor
model=RandomForestRegressor(n_estimators=500,max_depth=10,min_samples_split=2,min_samples_leaf=1,max_features="sqrt",random_state=42,n_jobs=-1)

model.fit(X_train,y_train)
y_pred=model.predict(X_test)
y_pred[:10]


# Feature Importance
It tells us which features are most influential and important.

In [ ]:
#feature importance
feature_importance=pd.DataFrame({
  "feature":X_train.columns,
  "Importance":model.feature_importances_
})
feature_importance=feature_importance.sort_values(by="Importance",ascending=False).reset_index(drop=True)
feature_importance

# Feature importance visualization

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(x="feature", y="Importance", data=feature_importance)
plt.title("Feature Importance")
plt.xticks(rotation=45)
plt.show()

# Model Evaluation
I use Mean Squared Error and R2_score to evaluate the model.

In [ ]:
#model evaluation
from sklearn.metrics import mean_squared_error, r2_score
MSE=mean_squared_error(y_test,y_pred)
R2=r2_score(y_test,y_pred)
RMSE=np.sqrt(MSE)
print("Mean Squared Error:",MSE)
print("Root Mean Squared Error:",RMSE)
print("R2_score:",R2)


Model evaluation tells us, the model achieved an RMSE of approximately 26 units, indicating that predictions deviate from actual values by about 26 units on average. Additionally, the R² score of 0.94 shows that the model explains 94% of the variance in sales, demonstrating strong predictive performance.

# Feature Forcasting
It predicts sales for the next 30 days.

In [ ]:
# Forecasting with proper feature updates
feature_days = 30
feature_prediction = []

# Get last known data
last_data = df.sort_values(by="Date").groupby(["Store ID","Product ID"]).tail(30)
current_data = last_data.copy()

for i in range(feature_days):

    next_day = current_data.groupby(["Store ID","Product ID"]).tail(1).copy()
    next_day["Date"] = next_day["Date"] + pd.Timedelta(days=1)

    next_day["Month"] = next_day["Date"].dt.month
    next_day["Is_Weekend"] = next_day["Date"].dt.weekday.apply(lambda x: 1 if x >= 5 else 0)

    next_day["Sales_lag_1"] = current_data.groupby(["Store ID","Product ID"])["Units Sold"].shift(0).iloc[-len(next_day):].values
    next_day["Sales_lag_7"] = current_data.groupby(["Store ID","Product ID"])["Units Sold"].shift(6).iloc[-len(next_day):].values
    next_day["Sales_lag_30"] = current_data.groupby(["Store ID","Product ID"])["Units Sold"].shift(29).iloc[-len(next_day):].values

    next_day[["Sales_lag_1","Sales_lag_7","Sales_lag_30"]] = next_day[["Sales_lag_1","Sales_lag_7","Sales_lag_30"]].fillna(0)

    rolling_window = current_data.groupby(["Store ID","Product ID"])["Units Sold"]

    next_day["Rolling_mean_3"] = rolling_window.apply(lambda x: x.tail(3).mean()).values
    next_day["Rolling_std_3"] = rolling_window.apply(lambda x: x.tail(3).std()).fillna(0).values

    next_day["Sales_lag_1_diff"] = next_day["Sales_lag_1"] - next_day["Sales_lag_7"]
    next_day["Sales_lag_1_vs_lag_7_diff"] = next_day["Sales_lag_1"] - next_day["Sales_lag_7"]
    next_day["Sales_lag_7_vs_lag_30_diff"] = next_day["Sales_lag_7"] - next_day["Sales_lag_30"]

    pred = model.predict(next_day[input_features])
    next_day["Units Sold"] = pred

    feature_prediction.append(next_day.copy())

    current_data = pd.concat([current_data, next_day], ignore_index=True)

feature_df = pd.concat(feature_prediction)
print(feature_df)

# Visualizaion on predictions
It visualizes the actual test vs the predicted with respect to the given data.

In [ ]:
#visualizations on pridictions
test_date=df["Date"].iloc[X_train.shape[0]:].reset_index(drop=True)

comparsion=pd.DataFrame({
  "Date":pd.to_datetime(test_date),
  "Actual":y_test.reset_index(drop=True),
  "Predicted":y_pred
})

agg_comparsion=comparsion.groupby("Date")[["Actual","Predicted"]].mean().reset_index()
print(agg_comparsion)
plt.figure(figsize=(12,6))
sns.lineplot(x="Date",y="Actual",data=agg_comparsion,color="blue",label="Actual")
sns.lineplot(x="Date",y="Predicted",data=agg_comparsion,color="red",label="Predicted")
plt.title("Average Daily Sales: Actual vs Predicted")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.legend()
plt.show()

# Feature forcasting visualization


In [ ]:
plt.figure(figsize=(12,6))
sns.lineplot(data=feature_df.groupby("Date")["Units Sold"].mean())
plt.title("Future 30 Days Sales Forecast")
plt.show()

# Interpertation of the graph
The graph illustrates a comparison between actual and predicted sales over time. The predicted values closely follow the actual values, indicating that the model performs well in capturing the overall sales pattern. However, the predictions appear smoother than the actual data, suggesting that the model is less sensitive to sudden fluctuations and extreme values. In particular, the model tends to underestimate peak sales and overestimate low sales, indicating limitations in capturing volatility. Despite this, the model demonstrates strong predictive performance overall, as it successfully tracks the general trend and variation in the data.

# Business Insights

- Sales show high daily fluctuations, indicating unstable demand.
- Discounts and promotions significantly influence demand.
- Peak sales are not fully captured, suggesting missing external factors.